In [3]:
%pip install pandas
%pip install numpy
%pip install seaborn
%pip install folium
%pip install dash
%pip install plotly
%pip install more-itertools

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


...


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [11]:
import dash
from dash import dcc, html
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

# Load data
data = pd.read_csv('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/d51iMGfp_t0QpO30Lym-dw/automobile-sales.csv')

app = dash.Dash(__name__)
app.title = "Automobile Dashboard"

dropdown_options = [
    {'label': 'Yearly Statistics', 'value': 'Yearly Statistics'},
    {'label': 'Recession Period Statistics', 'value': 'Recession Period Statistics'}
]

year_list = sorted(data['Year'].unique())

app.layout = html.Div([
    
    html.H1("Automobile Sales Dashboard", style={'textAlign': 'center'}),
    
    dcc.Dropdown(
        id='dropdown-statistics',
        options=dropdown_options,
        value='Yearly Statistics'
    ),
    
    dcc.Dropdown(
        id='select-year',
        options=[{'label': i, 'value': i} for i in year_list],
        value=year_list[0]
    ),
    
    html.Div(id='output-container')
])

# Enable/disable year dropdown
@app.callback(
    Output('select-year', 'disabled'),
    Input('dropdown-statistics', 'value')
)
def update_input_container(selected_statistics):
    return selected_statistics != 'Yearly Statistics'


# MAIN CALLBACK (FIXED)
@app.callback(
    Output('output-container', 'children'),
    [Input('dropdown-statistics', 'value'),
     Input('select-year', 'value')]
)
def update_output_container(selected_statistics, input_year):

    try:
        # 🔴 RECESSION
        if selected_statistics == 'Recession Period Statistics':

            recession_data = data[data['Recession'] == 1]

            if recession_data.empty:
                return [html.H3("No Recession Data Available")]

            # Chart 1
            yearly = recession_data.groupby('Year')['Automobile_Sales'].mean().reset_index()
            chart1 = dcc.Graph(figure=px.line(yearly, x='Year', y='Automobile_Sales'))

            # Chart 2
            avg = recession_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index()
            chart2 = dcc.Graph(figure=px.bar(avg, x='Vehicle_Type', y='Automobile_Sales'))

            # Chart 3
            exp = recession_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index()
            chart3 = dcc.Graph(figure=px.pie(exp, values='Advertising_Expenditure', names='Vehicle_Type'))

            # 🔥 IMPORTANT FIX (column name safe)
            unemp_col = 'Unemployment_Rate' if 'Unemployment_Rate' in data.columns else 'unemployment_rate'

            unemp = recession_data.groupby([unemp_col, 'Vehicle_Type'])['Automobile_Sales'].mean().reset_index()
            chart4 = dcc.Graph(
                figure=px.bar(unemp,
                              x=unemp_col,
                              y='Automobile_Sales',
                              color='Vehicle_Type')
            )

            return [
                html.Div([chart1, chart2], style={'display': 'flex'}),
                html.Div([chart3, chart4], style={'display': 'flex'})
            ]

        # 🔵 YEARLY
        elif selected_statistics == 'Yearly Statistics':

            yearly_data = data[data['Year'] == input_year]

            if yearly_data.empty:
                return [html.H3("No Data for Selected Year")]

            chart1 = dcc.Graph(
                figure=px.line(data.groupby('Year')['Automobile_Sales'].mean().reset_index(),
                               x='Year', y='Automobile_Sales')
            )

            chart2 = dcc.Graph(
                figure=px.line(yearly_data.groupby('Month')['Automobile_Sales'].sum().reset_index(),
                               x='Month', y='Automobile_Sales')
            )

            chart3 = dcc.Graph(
                figure=px.bar(yearly_data.groupby('Vehicle_Type')['Automobile_Sales'].mean().reset_index(),
                              x='Vehicle_Type', y='Automobile_Sales')
            )

            chart4 = dcc.Graph(
                figure=px.pie(yearly_data.groupby('Vehicle_Type')['Advertising_Expenditure'].sum().reset_index(),
                              values='Advertising_Expenditure', names='Vehicle_Type')
            )

            return [
                html.Div([chart1, chart2], style={'display': 'flex'}),
                html.Div([chart3, chart4], style={'display': 'flex'})
            ]

        return []

    except Exception as e:
        return [html.H3(f"Error: {str(e)}")]


if __name__ == '__main__':
    app.run(debug=True)